This notebook is based on the code of:
V. S. F. Garnot and L. Landrieu, “Panoptic Segmentation of Satellite Image Time Series with Convolutional Temporal Attention Networks,” in Proceedings of the IEEE/CVF International Conference on Computer Vision (ICCV), 2021, pp. 4872–4881, doi: 10.1109/ICCV48922.2021.00483.

In [25]:
import json
import os
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np

import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import DataLoader

import utaeT
from utils import iterate, save_results, prepare_output, pad_collate, get_ntrainparams, config_to_json, TreeDataset

In [ ]:
@dataclass
class ConfigPaths:
    # Dataset / paths
    path_base: Path = Path.cwd().parents[0] / "data" / "preprocessed data"
    paths_sentinel_data: list = None
    paths_tree_data: list = None
    paths_date_data: list = None
    paths_usable_data: list = None

    def __post_init__(self):
        self.paths_sentinel_data = [
            self.path_base / "satellite_patches_lin.npy",
            self.path_base / "satellite_patches_inn.npy",
            self.path_base / "satellite_patches_ham.npy",
            self.path_base / "satellite_patches_par.npy",
            self.path_base / "satellite_patches_vie.npy",
        ]

        self.paths_tree_data = [
            self.path_base / "tree_patches_lin.npy",
            self.path_base / "tree_patches_inn.npy",
            self.path_base / "tree_patches_ham.npy",
            self.path_base / "tree_patches_par.npy",
            self.path_base / "tree_patches_vie.npy",
        ]

        self.paths_date_data = [
            self.path_base / "date_patches_lin.npy",
            self.path_base / "date_patches_inn.npy",
            self.path_base / "date_patches_ham.npy",
            self.path_base / "date_patches_par.npy",
            self.path_base / "date_patches_vie.npy",
        ]

        self.paths_usable_data = [
            self.path_base / "usable_patches_lin.npy",
            self.path_base / "usable_patches_inn.npy",
            self.path_base / "usable_patches_ham.npy",
            self.path_base / "usable_patches_par.npy",
            self.path_base / "usable_patches_vie.npy",
        ]
    res_dir: Path = field(default_factory=lambda: Path.cwd() / "results_train")

In [ ]:
@dataclass
class Config:
    # Model parameters
    input_dim: int = 5
    encoder_widths: list = field(default_factory=lambda: [32, 32, 64, 64])
    decoder_widths: list = field(default_factory=lambda: [32, 32, 64, 64])

    str_conv_k: int = 4
    str_conv_s: int = 2
    str_conv_p: int = 1

    agg_mode: str = "att_group"
    encoder_norm: str = "batch"

    n_head: int = 4
    d_model: int = 32
    d_k: int = 8

    num_workers: int = 4
    rdm_seed: int = 1
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    def torch_device(self):
        return torch.device(self.device)
    
    # Loss parameters
    pos_weight = 2
    presence_threshold = 0.5
    count_loss_weight = 0.2
    count_loss_negative_mask = 0.1

    cache: bool = False

    # Training
    epochs: int = 5
    batch_size: int = 4
    lr: float = 1e-4

    # Image padding
    pad_value: float = 0.0
    padding_mode: str = "reflect"

    # Display results
    val_every: int = 1
    val_after: int = 0
    display_step: int = 5

    # Folds
    fold: int = 1

In [28]:
def weight_init(m):
    # Initialize weights for the model

    if isinstance(m, (nn.Conv2d, nn.Conv3d, nn.Conv1d, nn.ConvTranspose2d, nn.ConvTranspose3d, nn.ConvTranspose1d)):

        init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

        if m.bias is not None:
            init.zeros_(m.bias)

    elif isinstance(m, nn.Linear):
        init.xavier_uniform_(m.weight)

        if m.bias is not None:
            init.zeros_(m.bias)

    elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm3d, nn.BatchNorm1d)):
        init.ones_(m.weight)
        init.zeros_(m.bias)

In [29]:
def checkpoint(fold, log, config_paths):
    # create checkpoints to save best model for validation

    with open(os.path.join(config_paths.res_dir, "Fold_{}".format(fold), "trainlog.json"), "w") as outfile:
        json.dump(log, outfile, indent=4)

In [30]:
def split_into_8_blocks(N):
    # split all indices of data set into 5 blocks for train/val/test split

    indices = np.arange(N)

    # split into 8 (nearly) equal parts
    blocks = np.array_split(indices, 8)

    return blocks


In [31]:
def main(config, config_paths):
    # Main function calling other functions

    # set random seed and device
    np.random.seed(config.rdm_seed)
    torch.manual_seed(config.rdm_seed)
    device = torch.device(config.device)

    sentinel_list = []
    tree_list = []
    date_list = []
    usable_list = []
    
    # load data
    for p_s, p_t, p_d, p_u in zip(config_paths.paths_sentinel_data,config_paths.paths_tree_data,config_paths.paths_date_data,config_paths.paths_usable_data):
        sentinel_list.append(np.load(p_s, mmap_mode='r'))
        tree_list.append(np.load(p_t, mmap_mode='r'))
        date_list.append(np.load(p_d, mmap_mode='r'))
        usable_list.append(np.load(p_u, mmap_mode="r"))

    city_blocks = []
    # split into 8 blocks
    for s, t, d, u in zip(sentinel_list, tree_list, date_list, usable_list):
        blocks = split_into_8_blocks(len(s))
        city_blocks.append((s, t, d, u, blocks))

    fold_sequence = []

    # create training validation split
    for fold_id in range(5):
    
        train_x, train_y, train_d, train_u = [], [], [], []
        val_x, val_y, val_d = [], [], []
        for s, t, d, u, blocks in city_blocks:
            # TRAIN = 7 folds per city
            train_idx = np.concatenate([
                blocks[(fold_id + 0) % 8],
                blocks[(fold_id + 1) % 8],
                blocks[(fold_id + 2) % 8],
                blocks[(fold_id + 3) % 8],
                blocks[(fold_id + 5) % 8],
                blocks[(fold_id + 6) % 8],
                blocks[(fold_id + 7) % 8],
            ])
    
            val_idx = blocks[(fold_id + 4) % 8]
    
            train_x.append(s[train_idx])
            train_y.append(t[train_idx])
            train_d.append(d[train_idx])
            train_u.append(u[train_idx])
    
            val_x.append(s[val_idx])
            val_y.append(t[val_idx])
            val_d.append(d[val_idx])
    
        # merge all cities
        train_x = np.concatenate(train_x, axis=0)
        train_y = np.concatenate(train_y, axis=0)
        train_d = np.concatenate(train_d, axis=0)
        train_u = np.concatenate(train_u, axis=0)
    
        val_x = np.concatenate(val_x, axis=0)
        val_y = np.concatenate(val_y, axis=0)
        val_d = np.concatenate(val_d, axis=0)
    
        fold_sequence.append([(train_x, train_y, train_d, train_u),(val_x, val_y, val_d),])

    # create lists for test evaluation
    all_preds_folds = []
    all_targets_folds = []

    # define which folds should be trained
    if config.fold is not None:
        folds_to_run = [config.fold-1]
    else:
        folds_to_run = range(len(fold_sequence))
    
    # prepare output folders
    prepare_output(config_paths,config)

    for fold in folds_to_run:
        (train_x, train_y, train_d, train_u), (val_x, val_y, val_d) = fold_sequence[fold]

        dt_train = TreeDataset(train_x, train_y, train_d, train_u,augment=True)
        dt_val   = TreeDataset(val_x, val_y, val_d,augment=False)

        # create dataloaders using padding
        collate_fn = lambda x: pad_collate(x, pad_value=config.pad_value)
        train_loader = DataLoader(
            dt_train,
            batch_size=config.batch_size,
            shuffle=True,
            drop_last=True,
            collate_fn=collate_fn,
        )
        val_loader = DataLoader(
            dt_val,
            batch_size=config.batch_size,
            shuffle=False,
            drop_last=False,
            collate_fn=collate_fn,
        )

        print(f"Train {len(dt_train)}, Val {len(dt_val)}")

        # Model definition
        model = utaeT.UTAE(config)

        # get number of parameters
        config.N_params = get_ntrainparams(model)

        # safe config for future reproducibility
        with open(os.path.join(config_paths.res_dir, "conf.json"), "w") as file:
            json.dump(config_to_json(config), file, indent=4)

        # print info about the model
        print(model)
        print("TOTAL TRAINABLE PARAMETERS :", config.N_params)
        print("Trainable layers:")
        for name, p in model.named_parameters():
            if p.requires_grad:
                print(name)

        # get model on right device and initialize weights
        model = model.to(device)
        model.apply(weight_init)

        # Optimizer
        optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=2)

        # Training loop
        trainlog = {}
        best_loss = float("inf")
        for epoch in range(1, config.epochs + 1):
            print(f"EPOCH {epoch}/{config.epochs}")
            print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

            # Make one training iteration
            model.train()
            train_metrics, _, _ = iterate(
                model,
                data_loader=train_loader,
                config_train=config,
                optimizer=optimizer,
                mode="train"
            )

            # Check if model should be validated this epoch
            if epoch % config.val_every == 0 and epoch > config.val_after:

                # Validate model
                print("Validation . . . ")
                model.eval()
                val_metrics, val_preds, val_targets = iterate(
                    model,
                    data_loader=val_loader,
                    config_train=config,
                    optimizer=optimizer,
                    mode="val"
                )

                # Print loss and check if learning rate should be increased
                print(f"Loss: {val_metrics['val_loss']:.4f}")
                scheduler.step(val_metrics["val_loss"])

                # Create checkpoint
                trainlog[epoch] = {**train_metrics, **val_metrics}
                checkpoint(fold + 1, trainlog, config_paths)

                # Check if the current validation is the best, if so save the model
                if val_metrics["val_loss"] < best_loss:
                    best_loss = val_metrics["val_loss"]
                    torch.save(
                        {
                            "epoch": epoch,
                            "state_dict": model.state_dict(),
                            "optimizer": optimizer.state_dict(),
                        },
                        os.path.join(config_paths.res_dir, f"Fold_{fold + 1}","model.pth.tar"),
                    )
            else:
                trainlog[epoch] = {**train_metrics}
                checkpoint(fold + 1, trainlog, config_paths)

        # After training load and validate best epoch
        print("Validating best epoch . . .")
        model.load_state_dict(
            torch.load(
                os.path.join(
                    config_paths.res_dir, f"Fold_{fold + 1}", "model.pth.tar"
                )
            )["state_dict"]
        )
        model.eval()

        final_val_metrics, final_val_preds, final_val_targets = iterate(
            model,
            data_loader=val_loader,
            config_train=config,
            optimizer=optimizer,
            mode="test",
        )

        # Print and save results
        all_preds_folds.extend(final_val_preds)
        all_targets_folds.extend(final_val_targets)
        print(f"Loss {final_val_metrics['test_loss']:.4f}")
        save_results(fold + 1,final_val_metrics,config,config_paths,final_val_preds,final_val_targets)


In [32]:
if __name__ == "__main__":
    # Load configs and run main function

    config = Config()
    config_paths = ConfigPaths()
    main(config, config_paths)

Train 328, Val 47
UTAE(
  (in_conv): ConvBlock(
    (conv): ConvLayer(
      (conv): Sequential(
        (0): Conv2d(5, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (1): GroupNorm(8, 32, eps=1e-05, affine=True)
        (2): ReLU()
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (4): GroupNorm(8, 32, eps=1e-05, affine=True)
        (5): ReLU()
      )
    )
  )
  (down_blocks): ModuleList(
    (0): DownConvBlock(
      (down): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(8, 32, eps=1e-05, affine=True)
          (2): ReLU()
        )
      )
      (conv1): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(8, 32, eps=1e-05, affine=True)
          (2): ReL